In [1]:
!pip install -q -U transformers accelerate bitsandbytes peft datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 85.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 91.3 MB/s eta 0:00:00:00:01


# Non-Instructional Fine‑Tuning with a 4‑bit Quantized Model (QLoRA)

This notebook fine‑tunes TinyLlama-1.1B on raw text (e.g., Shakespeare).  
We use **4‑bit quantization** to fit the model in limited GPU memory,  
and **LoRA** so we only train a tiny fraction of the parameters.  
No instruction formatting – the model just learns to continue the text.

In [2]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,          # For 4‑bit quantization config
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling   # Handles causal LM data collation
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import os


## 1. Load the 4‑bit Quantized Model

We choose TinyLlama because it's small, open, and fits easily in Kaggle's GPU.  
The quantization config uses:
- **4‑bit NormalFloat** (NF4) – the recommended type for QLoRA.
- **Double quantization** – saves additional memory by quantizing the quantization constants.
- **bf16 compute dtype** – reduces memory further and speeds up math on newer GPUs.

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [4]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_use_double_quant=True,bnb_4bit_quant_type="nf4")
# Nested quantization for more memory savings

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

## 2. Prepare a Non‑Instructional Dataset

We'll read a raw text file (e.g., the complete works of Shakespeare),  
split it into fixed‑length chunks of tokens, and build a Hugging Face `Dataset`.  
This is "non‑instructional" because there are no prompts or responses –  
just a sequence of tokens where the model learns to predict the next one.

In [6]:

with open("/kaggle/input/datasets/kingburrito666/shakespeare-plays/alllines.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

block_size = 512  

# Tokenize the whole text, returning a 1D tensor of token IDs
tokenized = tokenizer(raw_text, return_tensors="pt", truncation=False)["input_ids"][0]

chunks = [tokenized[i : i + block_size] for i in range(0, len(tokenized), block_size)]
chunks = [chunk for chunk in chunks if len(chunk) == block_size]  

dataset = Dataset.from_dict({"input_ids": chunks})
# Split off 5% for evaluation
dataset = dataset.train_test_split(test_size=0.05)
print(dataset)  

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1497299 > 2048). Running this sequence through the model will result in indexing errors


DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 2777
    })
    test: Dataset({
        features: ['input_ids'],
        num_rows: 147
    })
})


In [7]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,   # Causal LM, not masked LM
)

## 3. Configure LoRA (Low‑Rank Adaptation)

LoRA adds small trainable matrices to the attention layers.  
We target `q_proj`, `v_proj`, `k_proj`, `o_proj` – the query, value, key, and output projections.  
With rank `r=8`, we train less than 0.2% of the total parameters.

In [8]:
# Cell 6: LoRA configuration
lora_config = LoraConfig(
    r=8,                # Rank of the low‑rank matrices. Higher = more capacity, more memory.
    lora_alpha=32,      # Scaling factor, typically 2x rank
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # Which layers to adapt
    lora_dropout=0.05,  # Dropout regularisation for the adapter
    bias="none",        # Don't train bias terms
    task_type="CAUSAL_LM",
)

In [9]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [ ]:
# Cell 7: Training arguments
training_args = TrainingArguments(
    output_dir="./tinyllama-shakespeare",  
    per_device_train_batch_size=4,        
    gradient_accumulation_steps=4,         
    num_train_epochs=3,                    
    learning_rate=2e-4,                  
    bf16=torch.cuda.is_bf16_supported(),  
    fp16=not torch.cuda.is_bf16_supported(), 
    logging_steps=10,                      
    save_strategy="epoch",                            
    report_to="none",                      
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
10,2.967608
20,2.893508
30,2.888436
40,2.876754
50,2.818944
60,2.821874
70,2.807284


In [ ]:
# Cell 8: Save the adapter
model.save_pretrained("shakespeare-lora-adapter")
tokenizer.save_pretrained("shakespeare-lora-adapter")

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
base_model = prepare_model_for_kbit_training(base_model)

# Load the trained LoRA adapter
from peft import PeftModel
model = PeftModel.from_pretrained(base_model, "shakespeare-lora-adapter")

In [ ]:
# Cell 10: Generate a stylized continuation
prompt = "To be or not to be,"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,     
    temperature=0.8,           
    do_sample=True,            
    repetition_penalty=1.1,    
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

## Summary

This notebook demonstrated:
- **4‑bit quantization** to massively reduce memory footprint.
- **QLoRA** to fine‑tune only a tiny adapter on a quantized model.
- **Non‑instructional fine‑tuning** on raw text, teaching the model a new style.

You can now replace the text file with any domain‑specific corpus (legal, medical, code)  
and adapt the same pipeline. Happy experimenting!